# Questões 11 e 12 - Agente clássico e agente baseado em LLM no planejamento de atividades

As duas questões foram resolvidas juntas porque usam o mesmo problema. A questão 11 pede a comparação entre um agente clássico e um agente baseado em LLM em um problema de planejamento. A questão 12 pede uma aplicação de agente baseado em LLM com interface em linguagem natural, inferência e explicabilidade. A aplicação escolhida para a questão 12 é o próprio assistente de planejamento da questão 11.

O notebook tem três partes: o agente clássico, o agente LLM e a comparação.

## 1. O problema

Um professor tem quatro horas livres, das 14h às 18h, e quatro atividades possíveis. O assistente deve montar um plano que respeite as restrições e aproveite o tempo da melhor forma.

As restrições são: a reunião acontece das 16h às 17h, as atividades não se sobrepõem, uma atividade iniciada deve ser concluída (a não ser que a divisão seja permitida), atividades urgentes e compromissos fixos têm preferência e toda atividade deixada de fora precisa de justificativa.

In [1]:
import json
import os

import pandas as pd

JANELA = (14 * 60, 18 * 60)

ATIVIDADES = {
    "A": {"nome": "Preparar aula de IA", "duracao": 90, "prioridade": "alta", "prazo": "hoje", "inicio_fixo": None},
    "B": {"nome": "Corrigir trabalhos", "duracao": 120, "prioridade": "media", "prazo": "amanha", "inicio_fixo": None},
    "C": {"nome": "Reunião de pesquisa", "duracao": 60, "prioridade": "alta", "prazo": "hoje", "inicio_fixo": 16 * 60},
    "D": {"nome": "Revisar artigo", "duracao": 60, "prioridade": "baixa", "prazo": "proxima semana", "inicio_fixo": None},
}


def hora(minutos):
    return f"{minutos // 60:02d}:{minutos % 60:02d}"


def minutos(texto):
    h, m = texto.split(":")
    return int(h) * 60 + int(m)


pd.DataFrame(ATIVIDADES).T[["nome", "duracao", "prioridade", "prazo"]]

,nome,duracao,prioridade,prazo
A,Preparar aula de IA,90,alta,hoje
B,Corrigir trabalhos,120,media,amanha
C,Reunião de pesquisa,60,alta,hoje
D,Revisar artigo,60,baixa,proxima semana


## 2. Agente clássico

**Tipo de agente**

Na taxonomia de Russell e Norvig, o agente adequado é o baseado em utilidade. Um agente baseado em objetivo não serve, porque não existe um estado final único que seja "o objetivo": há vários planos válidos e o agente precisa escolher o melhor. A função de utilidade é o que permite comparar os planos.

**Descrição PEAS**

Desempenho: soma da utilidade das atividades concluídas, sem violar restrições.

Ambiente: a agenda do professor, com as atividades, o horário disponível e os compromissos fixos.

Atuadores: a agenda proposta e as justificativas.

Sensores: a lista de atividades com duração, prioridade e prazo.

O ambiente é totalmente observável, determinístico, estático, discreto e de agente único. Por isso o problema pode ser resolvido por busca antes de qualquer ação.

**Formulação como busca**

Estado: o horário atual, o quanto falta de cada atividade e os blocos já agendados.

Estado inicial: 14h, nada agendado.

Ações: colocar uma atividade pendente no horário atual ou deixar o tempo livre até o próximo compromisso.

Teste de término: o horário chegou às 18h.

Utilidade: definida abaixo.

**Função de utilidade**

Cada atividade vale o produto de dois pesos, um da prioridade e um do prazo. Assim uma atividade de prioridade alta para hoje vale 9 e uma de prioridade baixa para a próxima semana vale 1.

In [2]:
PESO_PRIORIDADE = {"alta": 3, "media": 2, "baixa": 1}
PESO_PRAZO = {"hoje": 3, "amanha": 2, "proxima semana": 1}
BLOCO_MINIMO = 30


def valor(aid):
    a = ATIVIDADES[aid]
    return PESO_PRIORIDADE[a["prioridade"]] * PESO_PRAZO[a["prazo"]]


def minutos_feitos(plano):
    feitos = {aid: 0 for aid in ATIVIDADES}
    for b in plano:
        feitos[b["atividade"]] += b["fim"] - b["inicio"]
    return feitos


def utilidade(plano):
    return sum(valor(aid) * feito / ATIVIDADES[aid]["duracao"] for aid, feito in minutos_feitos(plano).items())


pd.Series({f"{aid} - {a['nome']}": valor(aid) for aid, a in ATIVIDADES.items()}, name="utilidade se concluída").to_frame()

,utilidade se concluída
A - Preparar aula de IA,9
B - Corrigir trabalhos,4
C - Reunião de pesquisa,9
D - Revisar artigo,1


**Verificador de restrições**

O verificador recebe um plano e devolve a lista de violações. Ele é independente do planejador. O agente clássico usa o verificador para conferir a própria resposta, e mais adiante o agente LLM usa o mesmo verificador como ferramenta.

In [3]:
def verificar_plano(plano, permitir_divisao=False):
    violacoes = []
    blocos = sorted(plano, key=lambda b: b["inicio"])
    for b in blocos:
        if b["atividade"] not in ATIVIDADES:
            violacoes.append(f"Atividade desconhecida: {b['atividade']}.")
        if b["inicio"] < JANELA[0] or b["fim"] > JANELA[1] or b["fim"] <= b["inicio"]:
            violacoes.append(f"Bloco {b['atividade']} {hora(b['inicio'])}-{hora(b['fim'])} fora do horário disponível.")
    for b1, b2 in zip(blocos, blocos[1:]):
        if b2["inicio"] < b1["fim"]:
            violacoes.append(f"Sobreposição entre {b1['atividade']} e {b2['atividade']}.")
    if violacoes:
        return violacoes

    feitos = minutos_feitos(plano)
    for aid, a in ATIVIDADES.items():
        meus = [b for b in blocos if b["atividade"] == aid]
        if a["inicio_fixo"] is not None:
            esperado = (a["inicio_fixo"], a["inicio_fixo"] + a["duracao"])
            if [(b["inicio"], b["fim"]) for b in meus] != [esperado]:
                violacoes.append(f"{a['nome']} deve ocorrer exatamente das {hora(esperado[0])} às {hora(esperado[1])}.")
            continue
        if feitos[aid] > a["duracao"]:
            violacoes.append(f"{a['nome']} recebeu {feitos[aid]} min, mais que a duração de {a['duracao']} min.")
        incompleta = 0 < feitos[aid] < a["duracao"]
        if not permitir_divisao and (len(meus) > 1 or incompleta):
            violacoes.append(f"{a['nome']} foi dividida ou ficou incompleta, e a divisão não foi permitida.")
        if a["prazo"] == "hoje" and feitos[aid] < a["duracao"]:
            violacoes.append(f"{a['nome']} tem prazo hoje e não foi concluída.")
    return violacoes

**Busca**

A busca é em profundidade e percorre todas as sequências de decisões possíveis. A cada horário livre, ela tenta cada atividade pendente e também a opção de deixar o tempo vazio. Os compromissos fixos são colocados automaticamente no horário deles. Ao chegar às 18h, o plano é verificado e, se for válido, sua utilidade é comparada com a do melhor plano encontrado até ali. Em caso de empate, vence o plano com menos tempo ocioso.

O problema é pequeno, então a busca completa é viável e a resposta é garantidamente ótima para a função de utilidade escolhida.

In [4]:
def planejar(permitir_divisao=False):
    fixos = sorted((a["inicio_fixo"], a["inicio_fixo"] + a["duracao"], aid) for aid, a in ATIVIDADES.items() if a["inicio_fixo"] is not None)
    flexiveis = {aid: a["duracao"] for aid, a in ATIVIDADES.items() if a["inicio_fixo"] is None}
    melhor = {"chave": (-1, 0), "plano": None}
    contagem = {"estados": 0, "planos_validos": 0}

    def explorar(t, falta, plano):
        contagem["estados"] += 1
        for ini, fim, aid in fixos:
            if t == ini:
                plano = plano + [{"atividade": aid, "inicio": ini, "fim": fim}]
                t = fim
        if t >= JANELA[1]:
            if not verificar_plano(plano, permitir_divisao):
                contagem["planos_validos"] += 1
                ocupado = sum(b["fim"] - b["inicio"] for b in plano)
                chave = (round(utilidade(plano), 6), ocupado)
                if chave > melhor["chave"]:
                    melhor.update(chave=chave, plano=plano)
            return
        limite = min([ini for ini, _, _ in fixos if ini > t] + [JANELA[1]])
        livre = limite - t
        for aid, resto in falta.items():
            if resto == 0:
                continue
            if resto <= livre:
                usar = resto
            elif permitir_divisao and livre >= BLOCO_MINIMO:
                usar = livre
            else:
                continue
            explorar(t + usar, {**falta, aid: resto - usar}, plano + [{"atividade": aid, "inicio": t, "fim": t + usar}])
        explorar(limite, falta, plano)

    explorar(JANELA[0], flexiveis, [])
    return {"plano": melhor["plano"], "utilidade": melhor["chave"][0], **contagem}

**Justificativas**

Para cada atividade que ficou de fora ou incompleta, o agente monta a justificativa a partir dos dados: quanto tempo a atividade exige, qual é o maior intervalo livre que sobrou e se o prazo permite adiar.

In [5]:
def intervalos_livres(plano):
    livres, t = [], JANELA[0]
    for b in sorted(plano, key=lambda b: b["inicio"]):
        if b["inicio"] > t:
            livres.append((t, b["inicio"]))
        t = b["fim"]
    if t < JANELA[1]:
        livres.append((t, JANELA[1]))
    return livres


def justificar(plano):
    feitos = minutos_feitos(plano)
    maior_livre = max([f - i for i, f in intervalos_livres(plano)] + [0])
    textos = []
    for aid, a in ATIVIDADES.items():
        falta = a["duracao"] - feitos[aid]
        if falta == 0:
            continue
        if feitos[aid] == 0:
            textos.append(
                f"{aid} - {a['nome']}: não realizada. Precisa de {a['duracao']} min seguidos e o maior intervalo livre que restou "
                f"tem {maior_livre} min. As atividades agendadas têm utilidade maior (esta vale {valor(aid)}). "
                f"O prazo é {a['prazo']}, então pode ser adiada."
            )
        else:
            textos.append(
                f"{aid} - {a['nome']}: feita em parte, {feitos[aid]} de {a['duracao']} min. Faltam {falta} min, "
                f"que ficam para depois, dentro do prazo ({a['prazo']})."
            )
    return textos


def mostrar(resultado, permitir_divisao=False):
    plano = resultado["plano"]
    tabela = pd.DataFrame(
        [{"início": hora(b["inicio"]), "fim": hora(b["fim"]), "atividade": f"{b['atividade']} - {ATIVIDADES[b['atividade']]['nome']}"} for b in plano]
    )
    print(tabela.to_string(index=False))
    for i, f in intervalos_livres(plano):
        print(f"Tempo livre: {hora(i)}-{hora(f)}")
    print(f"\nUtilidade total: {utilidade(plano):.2f}")
    print("Violações:", verificar_plano(plano, permitir_divisao) or "nenhuma")
    for texto in justificar(plano):
        print("Justificativa:", texto)

**Resultado sem permitir divisão**

In [6]:
sem_divisao = planejar(permitir_divisao=False)
mostrar(sem_divisao)
print(f"\nEstados visitados: {sem_divisao['estados']} | planos válidos avaliados: {sem_divisao['planos_validos']}")

início   fim               atividade
 14:00 15:30 A - Preparar aula de IA
 16:00 17:00 C - Reunião de pesquisa
 17:00 18:00      D - Revisar artigo
Tempo livre: 15:30-16:00

Utilidade total: 19.00
Violações: nenhuma
Justificativa: B - Corrigir trabalhos: não realizada. Precisa de 120 min seguidos e o maior intervalo livre que restou tem 30 min. As atividades agendadas têm utilidade maior (esta vale 4). O prazo é amanha, então pode ser adiada.

Estados visitados: 14 | planos válidos avaliados: 2


A preparação da aula ocupa o início da tarde, a reunião fica no horário fixo e a revisão do artigo entra na última hora. A correção dos trabalhos fica de fora: ela precisa de 120 minutos seguidos e o único intervalo desse tamanho é o das 14h às 16h, que precisa ser usado para a aula, que é urgente e vale mais. Sobram 30 minutos livres antes da reunião.

**Resultado permitindo divisão**

Se o professor aceitar dividir atividades, a resposta muda.

In [7]:
com_divisao = planejar(permitir_divisao=True)
mostrar(com_divisao, permitir_divisao=True)
print(f"\nEstados visitados: {com_divisao['estados']} | planos válidos avaliados: {com_divisao['planos_validos']}")

início   fim               atividade
 14:00 15:30 A - Preparar aula de IA
 15:30 16:00  B - Corrigir trabalhos
 16:00 17:00 C - Reunião de pesquisa
 17:00 18:00  B - Corrigir trabalhos

Utilidade total: 21.00
Violações: nenhuma
Justificativa: B - Corrigir trabalhos: feita em parte, 90 de 120 min. Faltam 30 min, que ficam para depois, dentro do prazo (amanha).
Justificativa: D - Revisar artigo: não realizada. Precisa de 60 min seguidos e o maior intervalo livre que restou tem 0 min. As atividades agendadas têm utilidade maior (esta vale 1). O prazo é proxima semana, então pode ser adiada.

Estados visitados: 40 | planos válidos avaliados: 12


Com a divisão permitida, a correção dos trabalhos ocupa os 30 minutos antes da reunião e a hora depois dela. São 90 dos 120 minutos, e os 30 restantes ficam para amanhã, dentro do prazo. Adiantar três quartos da correção vale mais que revisar o artigo inteiro, que tem prioridade baixa. Não sobra tempo ocioso.

**Limite do agente clássico**

O agente clássico acha o melhor plano, mas só entende a entrada no formato exato da tabela. Se o professor escrever "hoje não posso passar das 17h30" ou "a correção é mais importante do que parece", o agente não tem como usar essa informação sem que alguém altere o código. É nesse ponto que o LLM entra.

## 3. Agente baseado em LLM (questão 12)

**Especificação da aplicação**

A aplicação é um assistente de planejamento acadêmico. O professor escreve o pedido em português, do jeito que falaria com uma pessoa, e recebe a agenda com as justificativas.

O LLM não resolve o problema sozinho. Ele é o componente que entende o pedido, decide quais ferramentas chamar e redige a resposta. A busca do melhor plano e a conferência das restrições ficam com as ferramentas, que são as funções da seção anterior.

**Componentes**

Interface em linguagem natural: a função `conversar`, que recebe o pedido em texto livre e devolve a resposta em texto.

Inferência: um ciclo em que o LLM pensa, escolhe uma ação, recebe o resultado e pensa de novo. Esse padrão é conhecido como ReAct (raciocinar e agir).

Ferramentas: consultar a agenda, rodar o planejador, verificar um plano e calcular a utilidade de um plano.

Verificação obrigatória: o código só aceita a resposta final se o plano passar pelo verificador. Se o plano tiver violações, elas voltam para o LLM como uma nova observação e ele precisa corrigir.

Explicabilidade: todos os passos ficam registrados (pensamento, ação e resultado) e são mostrados ao usuário junto com a resposta. A justificativa final se apoia nos resultados das ferramentas e não só no texto do modelo.

**Modelo**

O agente usa a API da Anthropic pelo SDK oficial, com o modelo Claude Opus 5.5. A chave deve estar na variável de ambiente `ANTHROPIC_API_KEY`. Sem a chave, as células desta seção apenas avisam que não foram executadas.

In [8]:
import anthropic

MODELO = "claude-opus-5-5"
TEM_CHAVE = bool(os.environ.get("ANTHROPIC_API_KEY"))
cliente = anthropic.Anthropic() if TEM_CHAVE else None


def chamar_llm(sistema, mensagens):
    resposta = cliente.beta.messages.create(
        model=MODELO,
        max_tokens=16000,
        system=sistema,
        messages=mensagens,
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
    )
    if resposta.stop_reason == "refusal":
        raise RuntimeError("O modelo recusou o pedido.")
    return "".join(bloco.text for bloco in resposta.content if bloco.type == "text")

**Ferramentas**

Cada ferramenta é uma função Python que recebe argumentos simples e devolve um dicionário. Os horários trafegam como texto no formato HH:MM, que é mais natural para o modelo, e são convertidos para minutos aqui dentro.

In [9]:
def ler_plano(plano):
    return [{"atividade": b["atividade"], "inicio": minutos(b["inicio"]), "fim": minutos(b["fim"])} for b in plano]


def escrever_plano(plano):
    return [{"atividade": b["atividade"], "inicio": hora(b["inicio"]), "fim": hora(b["fim"])} for b in plano]


def f_consultar_agenda():
    return {
        "horario_disponivel": [hora(JANELA[0]), hora(JANELA[1])],
        "atividades": [
            {"id": aid, "nome": a["nome"], "duracao_min": a["duracao"], "prioridade": a["prioridade"], "prazo": a["prazo"],
             "horario_fixo": hora(a["inicio_fixo"]) if a["inicio_fixo"] is not None else None}
            for aid, a in ATIVIDADES.items()
        ],
    }


def f_planejar(permitir_divisao=False):
    r = planejar(permitir_divisao)
    return {"plano": escrever_plano(r["plano"]), "utilidade": r["utilidade"], "pendencias": justificar(r["plano"])}


def f_verificar_plano(plano, permitir_divisao=False):
    violacoes = verificar_plano(ler_plano(plano), permitir_divisao)
    return {"valido": not violacoes, "violacoes": violacoes}


def f_calcular_utilidade(plano):
    return {"utilidade": round(utilidade(ler_plano(plano)), 2)}


FERRAMENTAS = {
    "consultar_agenda": f_consultar_agenda,
    "planejar": f_planejar,
    "verificar_plano": f_verificar_plano,
    "calcular_utilidade": f_calcular_utilidade,
}

**Instruções do agente**

O prompt de sistema explica o papel do agente, lista as ferramentas e define o formato de resposta. A cada passo o modelo responde com um único objeto JSON, que contém o pensamento e uma ação, ou o pensamento e a resposta final.

In [10]:
SISTEMA = """Você é um assistente que organiza a agenda de um professor universitário.

Você trabalha em passos. Em cada passo responda com um único objeto JSON, sem nenhum texto fora dele.

Para usar uma ferramenta:
{"pensamento": "por que este passo é necessário", "acao": "nome_da_ferramenta", "argumentos": {}}

Para encerrar:
{"pensamento": "como chegou ao plano", "plano": [{"atividade": "A", "inicio": "14:00", "fim": "15:30"}], "permitir_divisao": false, "resposta_final": "texto para o professor"}

Ferramentas disponíveis:
- consultar_agenda(): devolve o horário disponível e as atividades com duração, prioridade, prazo e horário fixo.
- planejar(permitir_divisao): devolve o plano de maior utilidade que respeita as restrições. Use permitir_divisao=true apenas se o professor disser que aceita dividir atividades.
- verificar_plano(plano, permitir_divisao): confere um plano e lista as violações.
- calcular_utilidade(plano): devolve a utilidade total de um plano.

Regras:
- Sempre consulte a agenda antes de planejar. Não invente durações nem prazos.
- Se o professor impuser uma condição que o planejador não conhece, parta do plano do planejador, ajuste e confira com verificar_plano.
- Na resposta final, escreva em português simples, liste a agenda com horários e justifique cada atividade que ficou de fora ou incompleta.
- O plano da resposta final será conferido automaticamente. Se houver violação, você receberá a lista e deverá corrigir."""

**Ciclo do agente**

O ciclo roda até o modelo dar uma resposta final válida ou até o limite de passos. O resultado de cada ferramenta volta para o modelo como uma nova mensagem. Erros de formato e violações de restrição também voltam como observações, para o modelo se corrigir.

In [11]:
def extrair_json(texto):
    inicio, fim = texto.find("{"), texto.rfind("}")
    if inicio == -1 or fim == -1:
        raise ValueError("resposta sem JSON")
    return json.loads(texto[inicio : fim + 1])


def executar_agente(pedido, llm=None, max_passos=8):
    llm = llm or chamar_llm
    mensagens = [{"role": "user", "content": pedido}]
    registro = []
    for passo in range(1, max_passos + 1):
        texto = llm(SISTEMA, mensagens)
        mensagens.append({"role": "assistant", "content": texto})
        try:
            decisao = extrair_json(texto)
        except ValueError as erro:
            observacao = {"erro": f"Formato inválido ({erro}). Responda apenas com o objeto JSON."}
            registro.append({"passo": passo, "pensamento": "-", "acao": "resposta fora do formato", "resultado": observacao})
            mensagens.append({"role": "user", "content": json.dumps(observacao, ensure_ascii=False)})
            continue

        if "resposta_final" in decisao:
            divisao = bool(decisao.get("permitir_divisao", False))
            conferencia = f_verificar_plano(decisao.get("plano", []), divisao)
            registro.append({"passo": passo, "pensamento": decisao.get("pensamento", ""), "acao": "resposta final (conferida pelo verificador)", "resultado": conferencia})
            if conferencia["valido"]:
                return {"resposta": decisao["resposta_final"], "plano": decisao["plano"], "registro": registro}
            observacao = {"erro": "O plano final tem violações. Corrija.", "violacoes": conferencia["violacoes"]}
        else:
            nome, argumentos = decisao.get("acao"), decisao.get("argumentos") or {}
            try:
                observacao = FERRAMENTAS[nome](**argumentos)
            except Exception as erro:
                observacao = {"erro": f"Falha ao executar {nome}: {erro}"}
            registro.append({"passo": passo, "pensamento": decisao.get("pensamento", ""), "acao": f"{nome}({json.dumps(argumentos, ensure_ascii=False)})", "resultado": observacao})
        mensagens.append({"role": "user", "content": json.dumps(observacao, ensure_ascii=False)})
    return {"resposta": "Não foi possível chegar a um plano válido dentro do limite de passos.", "plano": None, "registro": registro}

**Interface e explicabilidade**

A função `conversar` é a interface com o usuário. Ela mostra a resposta do assistente e, em seguida, o registro dos passos. Esse registro é a parte de explicabilidade: o professor consegue ver quais dados foram consultados, qual ferramenta gerou o plano e o resultado da conferência final.

In [12]:
def resumir(resultado, limite=160):
    texto = json.dumps(resultado, ensure_ascii=False)
    return texto if len(texto) <= limite else texto[:limite] + "..."


def conversar(pedido, llm=None):
    if llm is None and not TEM_CHAVE:
        print("Célula não executada: defina a variável de ambiente ANTHROPIC_API_KEY e rode de novo.")
        return None
    saida = executar_agente(pedido, llm)
    print("PEDIDO\n" + pedido)
    print("\nRESPOSTA DO ASSISTENTE\n" + saida["resposta"])
    print("\nCOMO O ASSISTENTE CHEGOU A ESSA RESPOSTA")
    for r in saida["registro"]:
        print(f"\nPasso {r['passo']}")
        print("  Pensamento:", r["pensamento"])
        print("  Ação:      ", r["acao"])
        print("  Resultado: ", resumir(r["resultado"]))
    return saida

**Pedido 1: o cenário do enunciado**

In [13]:
PEDIDO_1 = (
    "Preciso preparar uma aula de Inteligência Artificial, corrigir trabalhos de alunos e participar de uma reunião de pesquisa. "
    "Tenho apenas quatro horas disponíveis hoje. A preparação da aula é urgente, mas a correção dos trabalhos precisa ser "
    "concluída até amanhã. Organize minhas atividades da melhor maneira possível."
)
saida_1 = conversar(PEDIDO_1)

Célula não executada: defina a variável de ambiente ANTHROPIC_API_KEY e rode de novo.


**Pedido 2: o professor aceita dividir atividades**

A informação nova vem em texto livre. O agente precisa perceber que ela corresponde ao parâmetro `permitir_divisao` do planejador.

In [14]:
PEDIDO_2 = PEDIDO_1 + " Se precisar, pode quebrar a correção dos trabalhos em mais de um bloco, não tem problema terminar o resto amanhã cedo."
saida_2 = conversar(PEDIDO_2)

Célula não executada: defina a variável de ambiente ANTHROPIC_API_KEY e rode de novo.


**Pedido 3: uma condição que o planejador não conhece**

O planejador não tem nenhum parâmetro para "sair mais cedo". O agente precisa adaptar o plano e conferir o resultado com o verificador.

In [15]:
PEDIDO_3 = PEDIDO_1 + " Só um detalhe: hoje preciso sair às 17h30 para buscar meu filho na escola."
saida_3 = conversar(PEDIDO_3)

Célula não executada: defina a variável de ambiente ANTHROPIC_API_KEY e rode de novo.


**LLM sem ferramentas**

Para comparar, o mesmo pedido é enviado ao modelo sem ferramentas e sem verificação. Ele recebe a tabela de atividades no texto e responde direto com um plano. O plano é então conferido pelo verificador e sua utilidade é comparada com a do plano ótimo.

In [16]:
def llm_sem_ferramentas(pedido):
    if not TEM_CHAVE:
        print("Célula não executada: defina a variável de ambiente ANTHROPIC_API_KEY e rode de novo.")
        return None
    sistema = (
        "Você organiza a agenda de um professor. Responda apenas com um objeto JSON no formato "
        '{"plano": [{"atividade": "A", "inicio": "14:00", "fim": "15:30"}], "explicacao": "..."}.'
    )
    contexto = "Dados da agenda: " + json.dumps(f_consultar_agenda(), ensure_ascii=False)
    resposta = extrair_json(chamar_llm(sistema, [{"role": "user", "content": pedido + "\n\n" + contexto}]))
    conferencia = f_verificar_plano(resposta["plano"])
    print("Plano proposto:", resposta["plano"])
    print("Explicação:", resposta["explicacao"])
    print("Conferência:", conferencia)
    if conferencia["valido"]:
        print("Utilidade:", f_calcular_utilidade(resposta["plano"])["utilidade"], "| ótimo:", sem_divisao["utilidade"])
    return resposta


direto = llm_sem_ferramentas(PEDIDO_1)

Célula não executada: defina a variável de ambiente ANTHROPIC_API_KEY e rode de novo.


## 4. Comparação (questão 11)

In [17]:
comparacao = pd.DataFrame(
    [
        ["Entrada", "Dados estruturados em formato fixo", "Texto livre em linguagem natural"],
        ["Representação do problema", "Explícita: estados, ações, restrições e utilidade no código", "Implícita nos pesos do modelo e no texto do prompt"],
        ["Como decide", "Busca no espaço de estados maximizando a utilidade", "Gera a próxima ação a partir do contexto e chama ferramentas"],
        ["Garantia de restrições", "Total, por construção", "Nenhuma por conta própria; depende do verificador"],
        ["Otimalidade", "Garantida para a utilidade definida", "Não garantida; obtida ao delegar para o planejador"],
        ["Repetibilidade", "Mesma entrada, mesma saída", "Pode variar entre execuções"],
        ["Situações não previstas", "Exige alterar o código", "Adapta-se a partir do texto do pedido"],
        ["Explicação", "Rastro exato da busca, pouco amigável", "Texto claro, mas pode não refletir o processo real"],
        ["Custo por consulta", "Milissegundos, sem custo", "Segundos e custo por token"],
        ["Esforço de construção", "Alto: modelar tudo antes", "Baixo para começar, alto para tornar confiável"],
    ],
    columns=["Aspecto", "Agente clássico (baseado em utilidade)", "Agente baseado em LLM"],
)
pd.set_option("display.max_colwidth", None)
comparacao

,Aspecto,Agente clássico (baseado em utilidade),Agente baseado em LLM
0,Entrada,Dados estruturados em formato fixo,Texto livre em linguagem natural
1,Representação do problema,"Explícita: estados, ações, restrições e utilidade no código",Implícita nos pesos do modelo e no texto do prompt
2,Como decide,Busca no espaço de estados maximizando a utilidade,Gera a próxima ação a partir do contexto e chama ferramentas
3,Garantia de restrições,"Total, por construção",Nenhuma por conta própria; depende do verificador
4,Otimalidade,Garantida para a utilidade definida,Não garantida; obtida ao delegar para o planejador
5,Repetibilidade,"Mesma entrada, mesma saída",Pode variar entre execuções
6,Situações não previstas,Exige alterar o código,Adapta-se a partir do texto do pedido
7,Explicação,"Rastro exato da busca, pouco amigável","Texto claro, mas pode não refletir o processo real"
8,Custo por consulta,"Milissegundos, sem custo",Segundos e custo por token
9,Esforço de construção,Alto: modelar tudo antes,"Baixo para começar, alto para tornar confiável"


**As duas concepções de agente**

No sentido clássico de Russell e Norvig, um agente é definido pela função que leva percepções a ações, e o projeto consiste em escolher a arquitetura certa para o ambiente (reativo, com modelo, com objetivo, com utilidade, com aprendizagem). O conhecimento sobre o problema é escrito de forma explícita pelo projetista: quais são os estados, o que cada ação faz e o que é um bom resultado. A inteligência vem do algoritmo de busca ou de inferência aplicado sobre essa representação.

No agente baseado em LLM, o centro é um modelo de linguagem treinado em texto. O projetista não escreve o modelo do mundo. Ele escreve instruções em linguagem natural e oferece ferramentas. O modelo interpreta o objetivo, decide os passos e usa as ferramentas para perceber e agir. O ciclo perceber, pensar e agir continua existindo, mas o "pensar" é uma geração de texto e não uma busca.

No problema da agenda, as duas soluções se encaixam bem nesses moldes. O agente clássico tem uma função de utilidade e uma busca que percorre todos os planos. O agente LLM lê o pedido do professor, consulta a agenda, chama o planejador e explica o resultado.

**Em que situações o agente LLM leva vantagem**

Quando a entrada é linguagem natural. O pedido do enunciado é um parágrafo de texto. O agente clássico não consegue ler esse parágrafo.

Quando aparecem informações que ninguém previu. No pedido 3, sair às 17h30 não é um parâmetro do planejador. O agente LLM consegue entender a condição e adaptar o plano, enquanto o agente clássico precisaria de uma alteração no código.

Quando o objetivo é vago ou incompleto. "Da melhor maneira possível" não define uma função de utilidade. O LLM consegue assumir uma interpretação razoável e pode perguntar ao usuário quando falta informação.

Quando é preciso combinar várias ferramentas e fontes de dados sem programar cada combinação.

Quando a resposta precisa ser comunicada a uma pessoa. A explicação em português é mais útil para o professor do que uma tabela com números de utilidade.

**O que ainda depende de mecanismos explícitos**

Verificação de restrições. Um LLM pode propor um plano com sobreposição ou com a reunião no horário errado e descrevê-lo como correto. Só um verificador externo garante que o plano respeita as regras. Por isso, nesta aplicação, a resposta final só é aceita depois da conferência.

Otimização e busca. Encontrar o melhor plano exige comparar todas as alternativas. O LLM não faz essa enumeração de forma confiável, e o erro cresce com o tamanho do problema. O planejador faz isso de forma exata.

Aritmética e contagem de tempo. Somar durações e conferir horários é simples para um programa e sujeito a erro para um modelo de linguagem.

Repetibilidade e auditoria. Em um sistema usado por uma universidade, a mesma entrada deve gerar a mesma agenda, e deve ser possível mostrar por que uma decisão foi tomada. O rastro do planejador e do verificador oferece isso. A explicação escrita pelo LLM é um texto plausível, que pode não corresponder ao que de fato determinou a resposta.

Planejamento de longo prazo. Em tarefas com muitos passos, o LLM tende a se perder ou a esquecer restrições ditas no começo. Estruturas explícitas de estado e de plano reduzem esse problema.

**Conclusão**

As duas abordagens se completam. O LLM é melhor na fronteira com o ser humano: entender o pedido, lidar com o imprevisto e explicar. Os mecanismos clássicos são melhores no núcleo do problema: buscar, otimizar e verificar. A aplicação deste notebook segue essa divisão: o LLM conduz a conversa e decide o que fazer, e as garantias vêm do planejador e do verificador.